In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # PyTorch Image Models (ResNeSt için şart)
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pandas as pd
import numpy as np

# Uyarıları gizlemek için (isteğe bağlı)
import warnings
warnings.filterwarnings("ignore")

In [7]:
import os
import torch

# --- KONFIGURASYON ---
# ResNeSt50 Modeli (timm kutuphanesindeki adi)
MODEL_NAME = 'resnest50d'

# Deney adi dosya ve klasor isimleri buna gore olacak
EXPERIMENT_NAME = "ResNeSt50_Baseline_MediumLarge_Run1"

# Hiperparametreler
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 8
DROPOUT_RATE = 0.4  # Overfitting'e karsi

# Dosya yollari
DATA_DIR = "../data/prepared-data"

# Sonuclarin kaydedilecegi yer
OUTPUT_DIR = f"../models/pytorch/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cihazi GPU'ya zorla
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA/GPU bulunamadi. GPU'da calismak icin NVIDIA surucusu, CUDA destekli PyTorch ve dogru kernel ortami gerekli."
    )

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

print(f"Cihaz: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Sayisi: {torch.cuda.device_count()}")
print(f"Model: {MODEL_NAME}")
print(f"Kayit Yeri: {OUTPUT_DIR}")

Cihaz: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU Sayisi: 1
Model: resnest50d
Kayit Yeri: ../models/pytorch/ResNeSt50_Baseline_MediumLarge_Run1


In [8]:
# ImageNet Normalize Değerleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri Dönüşümleri
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri Oluştur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderları Oluştur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=='train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"Sınıflar: {class_names}")
print(f"Eğitim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")

Sınıflar: ['dyed-lifted-polyps', 'dyed-resection-margins', 'esophagitis', 'normal-cecum', 'normal-pylorus', 'normal-z-line', 'polyps', 'ulcerative-colitis']
Eğitim Verisi: 5592
Validasyon Verisi: 1200


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

# Bu hücre tek başına çalıştırılırsa, eksik ayarları varsayılanlarla tamamla.
MODEL_NAME = globals().get("MODEL_NAME", "resnest50d")
NUM_CLASSES = globals().get("NUM_CLASSES", 8)
DROPOUT_RATE = globals().get("DROPOUT_RATE", 0.4)
LEARNING_RATE = globals().get("LEARNING_RATE", 0.001)
DEVICE = globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

def create_model():
    print(f"Model indiriliyor: {MODEL_NAME}...")
    # pretrained=True ile ImageNet ağırlıklarını alıyoruz
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE)
    return model

model = create_model()
model = model.to(DEVICE)

# Kayıp Fonksiyonu ve Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning Rate Scheduler (Plato görülürse öğrenme hızını düşür)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

print("Model GPU'ya yüklendi ve eğitime hazır.")

Model indiriliyor: resnest50d...
Model GPU'ya yüklendi ve eğitime hazır.


In [10]:
import os
import time
import torch

def train_model(model, criterion, optimizer, scheduler, num_epochs):
    required_globals = ["dataloaders", "dataset_sizes", "OUTPUT_DIR", "DEVICE"]
    missing = [name for name in required_globals if name not in globals()]
    if missing:
        raise RuntimeError(
            f"Eksik degisken(ler): {missing}. Lutfen once veri ve konfigürasyon hucrelerini calistirin."
        )

    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch Döngüsü
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Geçmişi Kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # En iyi modeli kaydet (Validasyon başarısına göre)
            if phase == 'val':
                scheduler.step(epoch_loss)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    save_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
                    torch.save(model.state_dict(), save_path)
                    print(f"En iyi Model! ({best_acc:.4f}) -> Kaydedildi.")

    time_elapsed = time.time() - since
    print(f'\nEğitim Tamamlandı: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En iyi Validasyon Doğruluğu: {best_acc:.4f}')

    # En iyi ağırlıkları geri yükle
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    return model, history

In [11]:
# --- BAŞLAT ---
required_vars = ["model", "criterion", "optimizer", "scheduler", "train_model"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        f"Eksik degisken(ler): {missing}. Lutfen once model/egitim hazirlik hucrelerini calistirin."
    )

num_epochs = int(globals().get("EPOCHS", 50))
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=num_epochs)

Epoch 1/50
----------


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(14, 5))

# Doğruluk Grafiği
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title(f'{MODEL_NAME} Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Kayıp Grafiği
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title(f'{MODEL_NAME} Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Kaydet ve Göster
plt.savefig(os.path.join(OUTPUT_DIR, 'training_graph.png'))
plt.show()
print(f"Grafikler kaydedildi: {os.path.join(OUTPUT_DIR, 'training_graph.png')}")

In [ ]:
print("\nTEST SETİ DEĞERLENDİRMESİ")

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 1. Classification Report (F1, Recall, Precision)
print("\nSınıflandırma Raporu:")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'))
plt.show()